In [62]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path("..")
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA.mkdir(parents=True, exist_ok=True)


In [63]:
retail_price = pd.read_csv(RAW_DATA / "retail_price.csv")

inventory = pd.read_csv(RAW_DATA / "retail_store_inventory.csv")

excel_file = RAW_DATA / "online_retail_II.xlsx"

sheets = pd.ExcelFile(excel_file).sheet_names

online_retail = pd.concat(
    [pd.read_excel(excel_file, sheet_name=s) for s in sheets], ignore_index=True
)

print("Retail Price:", retail_price.shape)
print("Inventory:", inventory.shape)
print("Online Retail:", online_retail.shape)


Retail Price: (676, 30)
Inventory: (73100, 15)
Online Retail: (1067371, 8)


In [64]:
retail_price.columns = (
    retail_price.columns.str.strip().str.lower().str.replace(" ", "_")
)


In [65]:
retail_price = retail_price.rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length",
    }
)


In [66]:
retail_price["month_year"] = pd.to_datetime(retail_price["month_year"], errors="coerce")


In [68]:
retail_price = retail_price.drop_duplicates()

retail_price = retail_price[
    (retail_price["qty"] >= 0)
    & (retail_price["unit_price"] > 0)
    & (retail_price["total_price"] >= 0)
].copy()

print("Shape:", retail_price.shape)
print("Missing values:")
display(retail_price.isnull().sum().sort_values(ascending=False).head(15))

Shape: (676, 30)
Missing values:


product_id                    0
product_category_name         0
month_year                    0
qty                           0
total_price                   0
freight_price                 0
unit_price                    0
product_name_length           0
product_description_length    0
product_photos_qty            0
product_weight_g              0
product_score                 0
customers                     0
weekday                       0
weekend                       0
dtype: int64

Cleaning retail store inventory

In [89]:
# Load inventory from raw data
inventory = pd.read_csv(RAW_DATA / "retail_store_inventory.csv")

# Standardize column names
inventory.columns = (
    inventory.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

# Convert date using the actual raw format: DD-MM-YYYY
inventory["date"] = pd.to_datetime(
    inventory["date"], format="%d-%m-%Y", errors="coerce"
)

# Convert numeric columns
numeric_columns = [
    "inventory_level",
    "units_sold",
    "units_ordered",
    "demand_forecast",
    "price",
    "discount",
    "competitor_pricing",
]

for col in numeric_columns:
    inventory[col] = pd.to_numeric(inventory[col], errors="coerce")

# Clean categorical columns
categorical_columns = [
    "store_id",
    "product_id",
    "category",
    "region",
    "weather_condition",
    "holiday_promotion",
    "seasonality",
]

for col in categorical_columns:
    inventory[col] = inventory[col].astype("string").str.strip()

# Remove duplicate rows
inventory = inventory.drop_duplicates()

# Remove impossible numeric values
inventory = inventory[
    (inventory["inventory_level"] >= 0)
    & (inventory["units_sold"] >= 0)
    & (inventory["units_ordered"] >= 0)
    & (inventory["price"] > 0)
].copy()

print("Inventory shape:", inventory.shape)
print("Missing dates:", inventory["date"].isna().sum())


Inventory shape: (73100, 15)
Missing dates: 0


In [90]:
inventory.columns = (
    inventory.columns.str.strip()
    .str.lower()
    .str.replace("/", "_")
    .str.replace(" ", "_")
)


In [91]:
inventory["date"] = pd.to_datetime(
    inventory["date"], format="%d-%m-%Y", errors="coerce"
)

categorical_columns = [
    "store_id",
    "product_id",
    "category",
    "region",
    "weather_condition",
    "holiday_promotion",
    "seasonality",
]

for col in categorical_columns:
    if col in inventory.columns:
        inventory[col] = inventory[col].astype("string").str.strip()


In [92]:
numeric_columns = [
    "inventory_level",
    "units_sold",
    "units_ordered",
    "demand_forecast",
    "price",
    "discount",
    "competitor_pricing",
]

for col in numeric_columns:
    if col in inventory.columns:
        inventory[col] = pd.to_numeric(inventory[col], errors="coerce")



inventory = inventory.drop_duplicates()

inventory = inventory[
    (inventory["inventory_level"] >= 0)
    & (inventory["units_sold"] >= 0)
    & (inventory["units_ordered"] >= 0)
    & (inventory["price"] > 0)
].copy()


In [93]:
print("Columns:")
for i, col in enumerate(inventory.columns):
    print(i, repr(col))


Columns:
0 'date'
1 'store_id'
2 'product_id'
3 'category'
4 'region'
5 'inventory_level'
6 'units_sold'
7 'units_ordered'
8 'demand_forecast'
9 'price'
10 'discount'
11 'weather_condition'
12 'holiday_promotion'
13 'competitor_pricing'
14 'seasonality'


Cleaning online_retail

In [94]:
online_retail.columns = (
    online_retail.columns.str.strip().str.lower().str.replace(" ", "_")
)

print(online_retail.columns.tolist())

online_retail = online_retail.rename(
    columns={
        "invoiceno": "invoice",
        "invoice_no": "invoice",
        "unitprice": "unit_price",
        "customerid": "customer_id",
        "customer_id": "customer_id",
    }
)

['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country', 'revenue']


In [95]:
print("Columns:")
for i, col in enumerate(online_retail.columns):
    print(i, repr(col))


Columns:
0 'invoice'
1 'stockcode'
2 'description'
3 'quantity'
4 'invoicedate'
5 'price'
6 'customer_id'
7 'country'
8 'revenue'


In [96]:
online_retail["invoice"] = online_retail["invoice"].astype("string").str.strip()

online_retail["stockcode"] = online_retail["stockcode"].astype("string").str.strip()

online_retail["description"] = online_retail["description"].astype("string").str.strip()

online_retail["country"] = online_retail["country"].astype("string").str.strip()

online_retail["invoicedate"] = pd.to_datetime(
    online_retail["invoicedate"], errors="coerce"
)

online_retail["quantity"] = pd.to_numeric(online_retail["quantity"], errors="coerce")

online_retail["price"] = pd.to_numeric(online_retail["price"], errors="coerce")


In [97]:
online_retail = online_retail.drop_duplicates()

online_retail = online_retail[
    online_retail["invoice"].notna()
    & online_retail["stockcode"].notna()
    & online_retail["invoicedate"].notna()
    & online_retail["quantity"].notna()
    & online_retail["price"].notna()
].copy()

online_retail = online_retail[
    ~online_retail["invoice"].str.upper().str.startswith("C")
].copy()

online_retail = online_retail[
    (online_retail["quantity"] > 0) & (online_retail["price"] > 0)
].copy()


online_retail["revenue"] = online_retail["quantity"] * online_retail["price"]


In [98]:
cleaned_datasets = {
    "retail_price": retail_price,
    "inventory": inventory,
    "online_retail": online_retail,
}

for name, df in cleaned_datasets.items():
    print("=" * 60)
    print(name)
    print("Rows:", f"{len(df):,}")
    print("Columns:", len(df.columns))
    print("Duplicates:", df.duplicated().sum())
    print("Missing values:", df.isnull().sum().sum())


retail_price
Rows: 676
Columns: 30
Duplicates: 0
Missing values: 0
inventory
Rows: 73,100
Columns: 15
Duplicates: 0
Missing values: 0
online_retail
Rows: 1,007,913
Columns: 9
Duplicates: 0
Missing values: 228488


In [100]:
retail_price.to_csv(PROCESSED_DATA / "retail_price_clean.csv", index=False)

inventory.to_csv(PROCESSED_DATA / "inventory_clean.csv", index=False)

online_retail.to_csv(PROCESSED_DATA / "online_retail_clean.csv", index=False)

print("Processed datasets saved.")


Processed datasets saved.
